In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Update held out alleles

 - Base models trained with `train_mhcprime_with_alleles_heldout.py`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "CLSBN26":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

In [ ]:
# imports
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.log_odds import *
from src.plotting import *

# misc
import warnings
warnings.filterwarnings("ignore")
device="cuda"

In [ ]:
# change gpu
# GPU_ID = 0
GPU_ID = 1
# GPU_ID = 2
# GPU_ID = 3

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")

Using GPU: NVIDIA H100 80GB HBM3 (ID: 1)


### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
from Data.mhc_pseudo_seqs import mhc_seq_dict
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### MS train and test sets

In [ ]:
# ms train and test sets
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

In [ ]:
# allele splits
allele_holdout_splits_path = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets" / "allele_holdout_splits.pkl"
allele_holdout_splits=load_dict_pickle(allele_holdout_splits_path)

In [ ]:
# full processing for running
ms_train_data=full_process_data(ms_train_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_train_data=create_seq_allele_column(ms_train_data)

ms_test_data=full_process_data(ms_test_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_test_data=create_seq_allele_column(ms_test_data)

### Load base models

In [ ]:
# setup tokenizer
tokenizer = AminoAcidTokenizer()

In [ ]:
# set base model params
base_model_params={
        'vocab_size': tokenizer.vocab_size,
        'embed_dim': 480,
        'fc_hidden_dims': [240, 120], 
        'fc_dropout': 0.1,
        'n_heads': 16,
        'max_peptide_len': 34,
        'max_mhc_len': 34,
        "n_peptide_layers": 2,
        "n_mhc_layers": 1,
        'n_concat_layers': 3,
        'add_pos_enc': True,
        'use_segment_embedder': True,
        'dim_feedforward_multiplier': 1,
        'debugging': False,
        'fc_output_dim': 1,
        'pooling': 'cls', # cls, mean, min, max, attn, for sequence ffnn head
        "ffnn_use_layernorm": False, 
        'use_feature_table': True,
        "feature_table": processed_feature_table,
        "sequence_embedder_feature_names": feature_names,
        'sequence_embedder_feature_mode': "concat",  # replace or concat
        'sequence_embedder_tokenizer': tokenizer,
        "peptide_start_pos": 1, 
        "mhc_start_pos": 35, 
        "pos_enc_type": "sincos", 
        "seed": 42,

        # peptide conformer
        "add_peptide_conformer": True, 
        "peptide_conformer_num_layers": 2,
        "peptide_conformer_n_heads": 16,
        "peptide_conformer_conv_kernel": 5,
        "peptide_conformer_dropout": 0.0,

        # mhc conformer
        "add_mhc_conformer": False, 
        "mhc_conformer_num_layers": 2,
        "mhc_conformer_n_heads": 16,
        "mhc_conformer_conv_kernel": 5,
        "mhc_conformer_dropout": 0.0,

        # fusion conformer parameters
        "add_fusion_conformer": True, 
        "fusion_conformer_num_layers": 1,
        "fusion_conformer_n_heads": 16,
        "fusion_conformer_conv_kernel": 7,
        "fusion_conformer_dropout": 0.0,
        "fusion_conformer_mask_bos": True,
        "ff_pre_post_swiglu": False,

        # legacy init
        "legacy_init_compat": True,
}

In [ ]:
# see models
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"
aho_dir = MODEL_CHECKPOINTS_DIR / "Alleles_Heldout_Models"
os.listdir(aho_dir)

['MHCPrime_Base_AHO_Split_3',
 'MHCPrime_Base_AHO_Split_7',
 'MHCPrime_Base_AHO_Split_2',
 'MHCPrime_Base_AHO_Split_8',
 'MHCPrime_Base_AHO_Split_6',
 'MHCPrime_Base_AHO_Split_9',
 'MHCPrime_Base_AHO_Split_4',
 'MHCPrime_Base_AHO_Split_5',
 'MHCPrime_Base_AHO_Split_10',
 'MHCPrime_Base_AHO_Split_1']

In [ ]:
%%time
# load base models
models={}
aho_dir = MODEL_CHECKPOINTS_DIR / "Alleles_Heldout_Models"
for i in range(1, 11):
    model_dir = f"{aho_dir}/MHCPrime_Base_AHO_Split_{str(i)}/Checkpoints"
    model_name=f"mhcprime_base_aho_split_{str(i)}"

    model=init_model(base_model_params, print_check=False, print_params=True)
    model.load_state_dict(torch.load(f"{model_dir}/model_final.pt"))
    model.eval()
    models[model_name]=model
    print("Model loaded from checkpoint...")
clear_all_gpu_memory()

[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 2

### PepCL updating

#### Run PepCL

 - Ran in `update_mhcprime_with_heldout_alleles.py`.

#### Inference

In [ ]:
# base models
base_and_ft_models=models.copy()

In [ ]:
%%time
# ms_test_data
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ms_test_data, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ms_test_data[model_name]=output_scores.detach().cpu().tolist()

3214300it [01:25, 37670.40it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (874.3 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:12<00:00,  7.93it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:11<00:00,  7.97it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:11<00:00,  7.95it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:15<00:00,  7.72it/s]


CPU times: user 8h 15min 26s, sys: 3min, total: 8h 18min 27s
Wall time: 23min 46s


In [ ]:
%%time
# load updated models
updated_models_output_dir=MODEL_CHECKPOINTS_DIR / "Alleles_Heldout_Updated_Models"
base_and_ft_models={}
for model_name in os.listdir(updated_models_output_dir):
    print(model_name)
    model=init_model(base_model_params, print_check=False, print_params=True)
    model.load_state_dict(torch.load(f"{updated_models_output_dir}/{model_name}"))
    model.eval()
    base_and_ft_models[model_name.removesuffix(".pt")]=model
    print("Model loaded from checkpoint...")
    
clear_all_gpu_memory()

mhcprime_ft_aho_split_10.pt
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
mhcprime_ft_aho_split_2.pt
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
mhcprime_pepcl_aho_split_8.pt
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
mhcprime_pepcl_aho_split_1.pt
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
m

In [ ]:
%%time
# ms_test_data
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ms_test_data, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ms_test_data[model_name]=output_scores.detach().cpu().tolist()

3214300it [01:22, 38970.45it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (874.3 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:11<00:00,  7.99it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:10<00:00,  8.02it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:10<00:00,  8.02it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:10<00:00,  8.03it/s]


CPU times: user 16h 11min 43s, sys: 6min 51s, total: 16h 18min 35s
Wall time: 46min 4s


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
# remove .pt suffic from any col names
ms_test_data=strip_pt_suffix(ms_test_data)

In [ ]:
%%time
"""
Save test set
"""
save_dfs=False
# save_dfs=True

if save_dfs:
    # ms test set
    datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
    ms_test_data.to_csv(f"{datasets_dir}/ms_test_data.csv")

CPU times: user 1min 30s, sys: 3.34 s, total: 1min 34s
Wall time: 1min 34s
